In [ ]:
import tensorflow as tf
import tensorflow_hub as hub
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

In [ ]:
# 1. Load a pre-trained Panoptic-DeepLab model from TensorFlow Hub
# Explore different Panoptic-DeepLab models at https://tfhub.dev/s?q=panoptic+deeplab
model_url = "https://tfhub.dev/sayakpaul/deeplabv3_plus_resnet50_panoptic_coco/1"
segmentation_model = hub.load(model_url)

In [ ]:
# 2. Load and preprocess the image
image_path = "path/to/your/image.jpg"  # Replace with the path to your image
img = cv2.imread(image_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
original_shape = img.shape[:2]

In [ ]:
# Resize the image to the model's expected input size
target_size = (512, 512)  # Common for this model
resized_img = cv2.resize(img, target_size)

In [ ]:
# Normalize the image
normalized_img = resized_img / 255.0
input_tensor = tf.expand_dims(normalized_img, 0)  # Add batch dimension

In [ ]:
# 3. Perform panoptic segmentation
predictions = segmentation_model(input_tensor)

In [ ]:
# 4. Extract segmentation masks and IDs
segmentation_masks = predictions['segmentation_masks'][0].numpy()
instance_ids = predictions['instance_ids'][0].numpy()
semantic_predictions = tf.argmax(predictions['semantic_predictions'][0], axis=-1).numpy().astype(np.uint8)

In [ ]:
# 5. Define color maps and class information (based on COCO Panoptic)
# You might need to adjust these based on the specific model
THING_CLASSES = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17, 18, 19, 20,
                 21, 22, 23, 24, 25, 27, 28, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40,
                 41, 42, 43, 44, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58,
                 59, 60, 61, 62, 63, 64, 65, 67, 70, 72, 73, 74, 75, 76, 77, 78, 79,
                 80, 81, 82, 84, 85, 86, 87, 88, 89, 90]
STUFF_CLASSES = [0, 12, 26, 29, 30, 45, 66, 68, 69, 71, 83, 91]
ALL_CLASSES = THING_CLASSES + STUFF_CLASSES
CLASS_NAMES = ['background', 'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus',
               'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'street sign',
               'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep',
               'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag',
               'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball', 'kite',
               'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket',
               'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana',
               'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza',
               'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'dining table',
               'toilet', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone',
               'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'book', 'clock',
               'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush', 'hair brush',
               'toilet paper', 'trash can', 'water bottle', 'other', 'floor', 'grass', 'walls',
               'windows', 'doors', 'stairs', 'ceiling', 'road', 'sidewalk', 'building', 'sky',
               'ground', 'sand', 'mountains', 'sea', 'lake', 'river', 'clouds', 'bridge', 'fence',
               'bush', 'tree', 'cactus', 'palm tree', 'field', 'path', 'dirt', 'gravel', 'rocks',
               'wood', 'stone', 'metal', 'plastic', 'paper', 'cardboard', 'cloth', 'leather',
               'rubber', 'glass', 'ice', 'snow', 'fog', 'smoke', 'fire', 'food', 'drink',
               'furniture', 'electronics', 'appliance', 'indoor', 'outdoor', 'animal', 'vehicle',
               'accessory', 'sports', 'kitchenware', 'foodstuff', 'beverage', 'utensil', 'fruit',
               'vegetable', 'cooked food', 'baked goods', 'canned goods', 'dairy product', 'meat',
               'fish', 'poultry', 'seafood', 'grain', 'pasta', 'sauce', 'soup', 'salad', 'dessert',
               'snack', 'candy', 'ice cream', 'other food', 'other drink', 'other furniture',
               'other electronics', 'other appliance', 'other indoor', 'other outdoor', 'other animal',
               'other vehicle', 'other accessory', 'other sports', 'other kitchenware',
               'other foodstuff', 'other beverage', 'other utensil', 'other fruit',
               'other vegetable', 'other cooked food', 'other baked goods', 'other canned goods',
               'other dairy product', 'other meat', 'other fish', 'other poultry',
               'other seafood', 'other grain', 'other pasta', 'other sauce', 'other soup',
               'other salad', 'other dessert', 'other snack', 'other candy', 'other ice cream']
CLASS_IDS_TO_NAMES = {i: name for i, name in enumerate(CLASS_NAMES)}

In [ ]:
# Create a color map for visualization
def create_panoptic_visualization(segmentation_mask, instance_ids, semantic_predictions):
    unique_instance_ids = np.unique(instance_ids)
    instance_colors = {id: np.random.randint(0, 256, 3) for id in unique_instance_ids if id != 0}
    colored_mask = np.zeros((segmentation_mask.shape[0], segmentation_mask.shape[1], 3), dtype=np.uint8)
    for i in range(segmentation_mask.shape[0]):
        for j in range(segmentation_mask.shape[1]):
            semantic_id = semantic_predictions[i, j]
            instance_id = instance_ids[i, j]
            if instance_id != 0:
                colored_mask[i, j] = instance_colors[instance_id]
            else:
                # Color stuff categories differently (you can define a color map for stuff)
                if semantic_id < len(CLASS_NAMES):
                    if semantic_id == 0:
                        colored_mask[i, j] = [0, 0, 0]  # Background
                    elif semantic_id in STUFF_CLASSES:
                        colored_mask[i, j] = [100, 100, 100] # Gray for stuff
                    else:
                        colored_mask[i, j] = [200, 200, 200] # Light gray for other things
    return colored_mask

In [ ]:
# 6. Create the panoptic visualization
panoptic_image = create_panoptic_visualization(segmentation_masks, instance_ids, semantic_predictions)
panoptic_image_resized = cv2.resize(panoptic_image, original_shape[::-1], interpolation=cv2.INTER_NEAREST)

In [ ]:
# 7. Display the results
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.imshow(img)
plt.title('Original Image')
plt.axis('off')
plt.subplot(1, 2, 2)
plt.imshow(panoptic_image_resized)
plt.title('Panoptic Segmentation')
plt.axis('off')
plt.show()